In [1]:
import numpy as np
import matplotlib.pyplot as plt
try:
     from dlroms import*
except:
     !pip install --no-deps git+https://github.com/NicolaRFranco/dlroms.git
     from dlroms import*

  Cloning https://github.com/NicolaRFranco/dlroms.git to /tmp/pip-req-build-pqlmey1y
  Running command git clone --filter=blob:none --quiet https://github.com/NicolaRFranco/dlroms.git /tmp/pip-req-build-pqlmey1y
  Resolved https://github.com/NicolaRFranco/dlroms.git to commit 1d6df1d34e882c082073358223d7957947632fa0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dlroms---Nicola-R-Franco: filename=dlroms_nicola_r_franco-2.4.4-py3-none-any.whl size=54704 sha256=73df54a2668abf5eb11dd50cf9cbb9bbe91638be989b6021259f596a271648cf
  Stored in directory: /tmp/pip-ephem-wheel-cache-2ivc7vso/wheels/37/2c/a9/c5ac2ca58529c8333ba7f8b315717dbb375099438969dc85ba
Successfully built dlroms---Nicola-R-Franco

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


ModuleNotFoundError: No module named 'torch'

# **Lab 7 - Neural networks**

Neural networks are function approximators constructed on top of composition of affine and nonlinear transformations. Since they can operate in between arbitrary dimensions, they are often considered more flexible than other classical techniques (polynomials, splines, wavelets, etc.).

Typically, a neural network model consists of some architecture $\Phi$ with $d$ trainable parameters, hereon collected in the vector $\boldsymbol{\theta}\in\mathbb{R}^{d}$. Mathematically speaking, a neural network architecture can be regarded as a map

$$\Phi:\Theta\times \mathcal{X}\to\mathcal{Y}$$

where $\Theta=\mathbb{R}^{d}$ is the space of trainable parameters, $\mathcal{X}$ is the input space, and $\mathcal{Y}$ the output space. In particular, for any fixed $\boldsymbol{\theta}\in\Theta$, the object $\Phi(\boldsymbol{\theta}, \cdot)$ defines as a map from $\mathcal{X}\to\mathcal{Y}$.
The so-called "training phase" consists in exploiting data to find an optimal $\boldsymbol{\theta}\in\Theta$, where optimality is typically defined in the least-square sense.
</br>
</br>
*Side note: to simplify, we can think of a linear regression model in 1d, where the "learnable" model would read: $\Phi(\boldsymbol{\theta},x)=\theta_{1}x+\theta_{2}$*.

## Dense layers and DNNs
Layers are the fundamental building block of neural networks. The most simple (and general) ones are the *dense* layers. Mathematically speaking, given an input dimension $m$, an output dimension $n$ and an activation function $\rho:\mathbb{R}\to\mathbb{R}$, a dense layer is a map of the form

$$L(\mathbf{v})=\rho\left(\mathbf{W}\mathbf{v}+\mathbf{b}\right)$$

where $\rho$ acts componentwise. The trainable parameters of a dense layer are its weight matrix, $\mathbf{W}$ and its bias vector, $\mathbf{b})$, respectively.

In [ ]:
# To construct dense layers, we use the class Dense from dlroms.dnns





In [ ]:
# We can create torch tensors using "dv" (which stands for "device", and it operates either on the CPU or GPU)
v = dv.tensor([[1, 1, 0]])




In general, neural network architectures can be constructed by combining multiple layers

### Example

Let's say that we wish to learn the map

$$ f: x\mapsto \sin(0.4x)e^{x/7}$$

from noisy samples $\{x_{i}, y_ {i}\}_{i=1}^{N_{\text{train}}}$, where $x_{i}\in[-10,10]$ and $y_{i}=f(x_{i})+\varepsilon_{i}$.
To this end, we shall construct a neural network architecture $\Phi=\Phi(\boldsymbol{\theta},x)$ and train it by minimizing the empirical loss
</br>

$$\mathcal{L}(\boldsymbol{\theta}):=\frac{1}{N}\sum_{i=1}^{N}\left|y_{i} - \Phi(\boldsymbol{\theta}, x_{i})\right|^{2}.$$



In [ ]:
# Ground truth
f = lambda x: np.sin(0.4*x)*(np.exp(x/7.0))

# Random sampling
np.random.seed(0)

n = 100
xdata = 20*np.random.rand(n)-10
ydata = f(xdata)

noise = 0.1*(2*np.random.rand(n) - 1)
ydata += noise

# Visualization
x = np.linspace(-10, 10, 1000)
y = f(x)
plt.figure(figsize = (5, 3))
plt.plot(x, y, label = 'Ground truth')
plt.plot(xdata, ydata, '.', label = 'Training data')
plt.legend()
plt.show()

In [ ]:
xdata.shape, ydata.shape

In [ ]:
# Transfering from numpy arrays to torch tensors




In [ ]:
from dlroms.roms import DFNN

# Neural network architecture design





In [ ]:
# Definition of the loss function





In [ ]:
# Training phase





In [ ]:
# Diagnostic
n_epochs = len(dnn.errors['Train'])
plt.figure(figsize = (5, 3))
plt.semilogx(dnn.errors['Train'], '-k', label = 'Train')
plt.semilogx(dnn.errors['Validation'], '--b', label = 'Validation')
plt.semilogx(dnn.errors['Test'], '--r', label = 'Test')
plt.xlabel('Epochs')
plt.ylabel('Error')
plt.axis([0.5, n_epochs, 0, 0.25])
plt.legend()
plt.show()

In [ ]:
# Freezing and evaluation
dnn.freeze()

plt.figure(figsize = (7, 4))
plt.plot(x, y, label = 'Ground truth')

y_network = dnn(dv.tensor(x).reshape(-1, 1)).reshape(-1).cpu().numpy()

plt.plot(x, y_network, label = 'Neural network')

plt.plot(xdata.cpu().numpy(), ydata.cpu().numpy(), '.r', label = 'Training data', markersize = 1)
plt.legend()
plt.show()

In [ ]:
## NB: Models can be saved using .save(...) and later loaded using .load(...)!
dnn.save("my_nn")

In [ ]:
error(ydata, dnn(xdata))

In [ ]:
dnn.unfreeze()
dnn.He()
error(ydata, dnn(xdata))

In [ ]:
dnn.load("my_nn")
error(ydata, dnn(xdata))

### Hands-on

<mark>**Exercise 1**</mark></br>
We want to learn the map

$$s:(x,y)\mapsto x - y^2$$

by relying on noisy samples $\{(x_i, y_i), z_i\}_{i=1}^{N}$, where $z_i\approx s(x_i, y_i)$. Implement and train a suitable neural network model using the data provided below. To this end, make sure to split the data between training and testing with a (50+20):30 ratio, the 20% being the portion devoted to the validation set.

In [2]:
mesh = fe.unitsquaremesh(50, 50)
Vh = fe.space(mesh, 'CG', 1)
x, y = fe.coordinates(Vh).T
s = lambda x, y: x - y**2

n = 100
np.random.seed(0)
xydata = np.random.rand(n, 2)
sdata = s(xydata[:, 0], xydata[:, 1])

noise = 0.02*(2*np.random.rand(n) - 1)
sdata += noise

clc()
plt.figure(figsize = (3, 3))
fe.plot(s(x, y), Vh, colorbar = True, shrink = 0.6)
plt.plot(xydata[:, 0], xydata[:, 1], '.k', label = 'Training points')
plt.axis([0, 1, 0, 1.2])
plt.legend(loc = 'upper left')
plt.show()

NameError: name 'fe' is not defined

In [ ]:
xydata.shape, sdata.shape

In [ ]:
xydata[0], sdata[0]

In [ ]:
# TODO: transfer numpy arrays to torch tensors, design NN model,train
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO

In [ ]:
# Diagnostic
n_epochs = len(dnn.errors['Train'])
plt.figure(figsize = (5, 3))
plt.semilogx(dnn.errors['Train'], '-k', label = 'Train')
plt.semilogx(dnn.errors['Validation'], '--b', label = 'Validation')
plt.semilogx(dnn.errors['Test'], '--r', label = 'Test')
plt.xlabel('Epochs')
plt.ylabel('Test error')
plt.axis([0.5, n_epochs, 0, 0.05])
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize = (7, 3))
plt.subplot(1, 2, 1)
fe.plot(s(x, y), Vh, colorbar = True, vmin = -1, vmax = 1)
plt.title("Ground truth")

plt.subplot(1, 2, 2)
xytensor = dv.tensor(fe.coordinates(Vh))
fe.plot(dnn(xytensor).reshape(-1), Vh, colorbar = True, vmin = -1, vmax = 1)
plt.title("Neural network")

plt.show()

## Convolutional neural networks

Convolutional neural networks are particular architectures tailored for handling high-dimensional inputs, coming in form of *channelled-data* (images in 2D, time series in 1D, etc.). They were introduced to mitigate the number of trainable parameters, as dense layers typically resulted in very complex architectures.

In brief, convolutional layers use a "shared" bias value (instead of a whole vector) and rely on a convolutional kernels rather than weight matrices (the action is still linear, but it can be described by far less parameters).
</br></br>
*Note: differently from dense layers, convolutional layers can operate on inputs with different dimensions, as long as their shapes are consistent.*

In [ ]:
from dlroms.dnns import Conv2D

L = Conv2D(window = 2, channels = (1, 1))
print(L.w())
print()
print(L.b())

In [ ]:
# NB: 2D convolutional layers expect inputs to be of the form Nsamples x Nchannels x Xdim x Ydim
from dlroms.dnns import Reshape

nn = Reshape(1, 10, 10) + Conv2D(window = 2, channels = (1, 4)) + Conv2D(window = 2, channels = (4, 2)) + Reshape(-1)
nn.dims()

### Hands-on

The following piece of code grants you access to a dataset $\{\mathbf{D}_{i}, \mathbf{c}_{i}\}_{i=1}^{N}$ where, for each $i=1,\dots,N$,

- $\mathbf{D}_{i}$ is a $51\times51$ matrix representing an image of a circle within the unit square;

- $\mathbf{c}_{i}\in\mathbb{R}^{2}$ is the center of the circle in $\mathbf{D}_{i}$.

In [ ]:
import gdown
gdown.download(id = "14vdopFVYG-TPjL7fyfhg482FslesZ2x2", output = "circles.npz")
dataset = np.load("circles.npz")

D, c = dataset["images"], dataset["centers"]

In [ ]:
print("Examples extracted from the data at hand:\n")
plt.figure(figsize = (8, 3))
for i, k in enumerate([0, -1, 5]):
  plt.subplot(1,3,i+1)
  plt.imshow(D[k], origin = 'lower')
  plt.axis("off")
  plt.title("$c$ = (%.2f, %.2f)." % tuple(c[k]))

<mark>**Exercise 2**</mark></br>
Learn the map $\mathbf{D}\mapsto\mathbf{c}$ using a suitable neural network model. To this end, use a combination of convolutional and dense layers.
</br></br>
*Side question:* at a continuous level, the ground truth relationship between input and output can be viewed as a map $\mathcal{F}:L^{1}(\Omega)\to\mathbb{R}^{2}$, with $\Omega=(0,1)^2$. Can you provide an analytic expression for $\mathcal{F}$? What is the regularity of $\mathcal{F}$?

In [ ]:
# TODO: transfer numpy arrays to torch, design CNN
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO

In [ ]:
def loss(true, pred):
    return (true-pred).pow(2).sum(axis = -1).mean()

def error(true, pred):
    return (true-pred).abs().sum(axis = -1).mean()

In [ ]:
# TODO train CNN
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO

In [ ]:
cnn.freeze()
n_epochs = len(cnn.errors['Train'])
plt.figure(figsize = (5, 3))
plt.semilogx(cnn.errors['Train'], '-k', label = 'Train')
plt.semilogx(cnn.errors['Validation'], '--b', label = 'Validation')
plt.semilogx(cnn.errors['Test'], '--r', label = 'Test')
plt.xlabel('Epochs')
plt.ylabel('Test error')
plt.legend()
plt.show()

In [ ]:
predicted = cnn(D).cpu().numpy()
plt.figure(figsize = (12, 3))
for j in range(4):
    image = D[-j-1].cpu().numpy()
    center = 51*predicted[-j-1]
    plt.subplot(1,4,j+1)
    plt.imshow(image, origin = 'lower')
    plt.plot(*center, 'x', color = 'red', label = 'Predicted')
    plt.legend()
    plt.axis("off")

<mark>**Exercise 3**</mark></br>
The following piece of code gives you access to another dataset, analogous to the previous one, where the "circles" have been substituted with "squares". Test your previous model (the one trained in Ex. 2) on this new dataset: does it work?

In [ ]:
gdown.download(id = "1-2G3b7hCCeqNE3Kr6oPuNW9_VX1hdWgV", output = "squares.npz")
squares_dataset = np.load("squares.npz")
squares, centers = squares_dataset["images"], squares_dataset["centers"]

In [ ]:
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO
# TODO